<h4>Understanding ratings table</h4>

In [7]:
import pandas as pd

base = "dataset/ml-100k"

ratings = pd.read_csv(
    f"{base}/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

print("Ratings:")
print(ratings.head())
print(ratings.shape)

Ratings:
   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
(100000, 4)


In [19]:
print("*" * 60)
print("Ratings table infos")
print("*" * 60)
ratings.info()

print("*" * 60)
print("Ratings table description")
print("*" * 60)
print(ratings.describe())

print("*" * 60)
print("Ratings distribution")
print("*" * 60)
print("Number of ratings:", len(ratings))
print(ratings["rating"].value_counts().sort_index())

************************************************************
Ratings table infos
************************************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB
************************************************************
Ratings table description
************************************************************
            user_id       movie_id         rating     timestamp
count  100000.00000  100000.000000  100000.000000  1.000000e+05
mean      462.48475     425.530130       3.529860  8.835289e+08
std       266.61442     330.798356       1.125674  5.343856e+06
min         1.00000       1.000000       1.000000  8.747247e+08
25%      

<h4>Implicit feedback</h4>

We turn explicit ratings into a simpler signal: rating 4 or 5 means the user liked the movie. Ratings below 4 are not kept as positive interactions.

In [20]:
ratings_implicit = ratings.copy()
ratings_implicit["liked"] = (ratings_implicit["rating"] >= 4).astype(int)

ratings_implicit[["user_id", "movie_id", "rating", "liked"]].head()

,user_id,movie_id,rating,liked
0,196,242,3,0
1,186,302,3,0
2,22,377,1,0
3,244,51,2,0
4,166,346,1,0


<h4>Keep positive interactions</h4>

For implicit recommendation data, we usually keep only the interactions where the user showed positive interest.

In [21]:
positive_interactions = ratings_implicit[ratings_implicit["liked"] == 1].copy()
positive_interactions = positive_interactions[["user_id", "movie_id", "liked", "timestamp"]]

print("Original ratings:", len(ratings))
print("Positive interactions:", len(positive_interactions))

positive_interactions.head()

Original ratings: 100000
Positive interactions: 55375


,user_id,movie_id,liked,timestamp
5,298,474,1,884182806
7,253,465,1,891628467
11,286,1014,1,879781125
12,200,222,1,876042340
16,122,387,1,879270459


<h4>Sparsity</h4>

Sparsity tells us how empty the user-movie matrix is. If there are many possible user-movie pairs but only a few observed positive interactions, sparsity is high.

In [22]:
num_users = ratings["user_id"].nunique()
num_movies = ratings["movie_id"].nunique()
possible_interactions = num_users * num_movies

positive_density = len(positive_interactions) / possible_interactions
positive_sparsity = 1 - positive_density

print("Users:", num_users)
print("Movies:", num_movies)
print("Possible user-movie pairs:", possible_interactions)
print(f"Positive density: {positive_density:.4%}")
print(f"Positive sparsity: {positive_sparsity:.4%}")

Users: 943
Movies: 1682
Possible user-movie pairs: 1586126
Positive density: 3.4912%
Positive sparsity: 96.5088%


<h4>Merge with movie titles</h4>

Movie ids are useful for computation, but titles are easier for humans to read. We merge the positive interactions with the movie table using movie_id.

In [38]:
movie_columns = [
    "movie_id", "title", "release_date", "video_release_date", "imdb_url",
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]

movies = pd.read_csv(
    f"{base}/u.item",
    sep="|",
    encoding="latin-1",
    names=movie_columns
)

positive_with_titles = positive_interactions.merge(
    movies[["movie_id", "title"]],
    on="movie_id",
    how="left"
)

positive_with_titles.head()

,user_id,movie_id,liked,timestamp,title
0,298,474,1,884182806,Dr. Strangelove or: How I Learned to Stop Worr...
1,253,465,1,891628467,"Jungle Book, The (1994)"
2,286,1014,1,879781125,Romy and Michele's High School Reunion (1997)
3,200,222,1,876042340,Star Trek: First Contact (1996)
4,122,387,1,879270459,"Age of Innocence, The (1993)"
